# 02 · Split the SMP "usual" into train / eval / test

This notebook now does two main tasks:
* Create train / eval / test splits for the SMP2020-EWECT usual subset
* Build the matched slang vs. non-slang datasets needed for RQ1: does slang cause more sentiment loss in translation from Chinese to English?


### Rules for the split
1. Only the **usual** subset
2. Keep identical text in the same split so duplicate posts cannot leak across splits
3. Stratify by **emotion label × slang/no-slang**.
4. Route HADQAET-overlapping texts away from SMP eval/test
> **Important:** putting HADQAET-overlapping posts in the SMP `train` partition here only keeps them out of the SMP eval/test experiment sets.  
> Notebook 04 must still **exclude `in_hadqaet == True` rows from mBERT training** if HADQAET will be used as a human validation benchmark.

### RQ1 matched-control design
For every **unique slang-containing post** in eval/test, choose one non-slang control that:
- comes from the same split
- has the same gold emotion label
- has the closest possible Chinese character length
- is not reused for another slang post


### Outputs
- `data/processed/usual_splits.csv`
- `data/processed/rq1_matched_eval.csv`
- `data/processed/rq1_matched_test.csv`
- `data/processed/rq1_matched_controls.csv`
- `data/processed/slang_match_audit_sample.csv`

## 0. Setup

In [ ]:
import os
import random
import pandas as pd

In [ ]:
# CAN CHANGE THESE
eval_fraction = 0.10
test_fraction = 0.10 # everything else (~80%) goes to train
random_seed = 266

# Folders (same logic as notebook 01)
current_folder = os.getcwd()
if os.path.basename(current_folder) == "notebooks":
    project_root = os.path.dirname(current_folder)
else:
    project_root = current_folder

processed_folder = os.path.join(project_root, "data", "processed")
print("Saving to:", processed_folder) #including this because I messed up the save location before

Saving to: /Users/cierrajagan/github/W266-slang-emotion-mt/data/processed


## 1. Load the data

In [ ]:
smp = pd.read_csv(os.path.join(processed_folder, "smp2020_ewect.csv"))
hadqaet = pd.read_csv(os.path.join(processed_folder, "hadqaet_annotated.csv"))
cheer = pd.read_csv(os.path.join(processed_folder, "cheer_lexicon.csv"))

# Keep only the 'usual' subset.
usual = smp[smp["subset"] == "usual"].copy()

# Rename SMP's original split column so it doesn't clash with our new one (not using their defined split because of the HADQAET overlap)
usual = usual.rename(columns={"split": "orig_split"})
print("Usual posts:", len(usual))
print("HADQAET rows:", len(hadqaet))
print("CHEER buzzwords:", len(cheer))

Usual posts: 34768
HADQAET rows: 5538
CHEER buzzwords: 1127


In [ ]:
# Remove empty posts
keep_rows = []
for text in usual["text"]:
    # pd.isna gets the cells pandas read as missing and strip() == "" gets the blank ones
    if pd.isna(text) or str(text).strip() == "":
        keep_rows.append(False)
    else:
        keep_rows.append(True)

rows_before = len(usual)
usual = usual[keep_rows].copy() #using .copy() because I always get messed up when making copies of data with python's reference system
print("Removed: ", rows_before - len(usual))
print(len(usual), " posts left")

Removed:  2
34766  posts left


## 2. Mark HADQAET posts

Notebook 01 saved the matching SMP id for each HADQAET row in the smp_post_id column. We:
- collect those ids
- mark every usual post whose text matches one of those post (also catches repeated copies of the same text)

In [ ]:
# The set of SMP post ids that HADQAET matched
hadqaet_post_ids = set()
for post_id in hadqaet["smp_post_id"]:
    if pd.isna(post_id):
        continue
    hadqaet_post_ids.add(post_id)

# The texts of those posts
hadqaet_texts = set()
for index, row in usual.iterrows():
    if row["post_id"] in hadqaet_post_ids:
        hadqaet_texts.add(row["text"].strip()) #strip the text for a better comparison

# mark every post whose text is one of those texts
in_hadqaet = []
for text in usual["text"]:
    cleaned_text = text.strip()# remove spaces
    if cleaned_text in hadqaet_texts:# check if this text one of the HADQAET texts
        in_hadqaet.append(True)
    else:
        in_hadqaet.append(False)
usual["in_hadqaet"] = in_hadqaet #add to text table

print(f"HADQAET post ids found in usual: {len(hadqaet_post_ids)}")
print(f"Rows marked as HADQAET (including repeats): {sum(in_hadqaet)}" )

HADQAET post ids found in usual: 5510
Rows marked as HADQAET (including repeats): 5977


## 3. Mark posts that (probably) contain slang

Same simple check as notebook 01: is a CHEER buzzword anywhere in the text? Only using it here to balance slang across the splits.

In [ ]:
buzzwords = list(cheer["buzzword"])

has_slang = []
for text in usual["text"]:
    found = False
    for word in buzzwords:
        if word in text:
            found = True
            break                    # one buzzword is enough; stop checking this post
    has_slang.append(found)

usual["has_slang_simple"] = has_slang
print("Posts with a buzzword:", sum(has_slang), "of", len(usual))

Posts with a buzzword: 885 of 34766


## 4. Group identical text

Each unique text becomes one "group" that carries all of its rows with it (since some text is repeated).

Repeated text can have two different labels (maybe annotators disagreed). For balancing purposes (maintaing the emotion split percentages) we give each group the label of its first row, (but the origianl labels stay unchanged in the saved file)

In [ ]:
groups = {} # text -> information about that text

for index, row in usual.iterrows():
    text = row["text"].strip()

    if text not in groups: # First time we see this text: create its group
        groups[text] = {
            "label": row["label"],
            "has_slang": row["has_slang_simple"],
            "in_hadqaet": row["in_hadqaet"],
            "row_count": 0,
        }

    groups[text]["row_count"] = groups[text]["row_count"] + 1

print("Rows:", len(usual))
print("Unique texts (groups):", len(groups))

Rows: 34766
Unique texts (groups): 33605


## 5. Do the split

SPlitting based on label (emotion) and slang/no-slang (combination of label + slang/no-slang, e.g. "angry + slang"). I googled it and these are called "strata":

1. Count its unique texts and work out how many should go to eval and test (10% each).
2. Send every HADQAET text in that split to train.
3. Shuffle the remaining texts and fill eval, then test. Whatever is left goes to train.

Because HADQAET texts can only go to train, eval and test are filled entirely from non-HADQAET texts.

In [ ]:
# A "stratum" is a group of texts that have the same label AND contain/don't contain slang,
# for example "angry + slang" or "happy + no slang".
strata = {} #stratum name -> list of texts in that stratum

for text in groups:
    # Look up this text's label (ex "angry")
    label = groups[text]["label"]

    # Turn True/False into a readable word
    if groups[text]["has_slang"]:
        slang_part = "slang"
    else:
        slang_part = "no slang"

    # Build the stratum name, e.g. "angry + slang"
    stratum_name = label + " + " + slang_part

    # If this is the first text in this stratum, start an empty list for it
    if stratum_name not in strata:
        strata[stratum_name] = []

    # Add the text to its stratum's list
    strata[stratum_name].append(text)

# Show how many texts ended up in each stratum
for stratum_name in sorted(strata):
    print(stratum_name.ljust(22), len(strata[stratum_name]), "texts")

angry + no slang       9211 texts
angry + slang          346 texts
fear + no slang        1391 texts
fear + slang           55 texts
happy + no slang       6559 texts
happy + slang          183 texts
neutral + no slang     7085 texts
neutral + slang        44 texts
sad + no slang         6052 texts
sad + slang            141 texts
surprise + no slang    2482 texts
surprise + slang       56 texts


In [ ]:
random.seed(random_seed)

split_of_text = {} #text -> "train", "eval", or "test"

# Go through the strata in alphabetical order
stratum_names_in_order = sorted(strata)

for stratum_name in stratum_names_in_order:
    # Sort the texts first so starting order is always the same
    texts_in_stratum = sorted(strata[stratum_name])

    # How many texts should go to eval and to test?
    total_count = len(texts_in_stratum)
    eval_count = round(total_count * eval_fraction)
    test_count = round(total_count * test_fraction)

    # Separate HADQAET texts from all other texts
    hadqaet_texts_here = []
    other_texts_here = []
    for text in texts_in_stratum:
        if groups[text]["in_hadqaet"] == True:
            hadqaet_texts_here.append(text)
        else:
            other_texts_here.append(text)

    # Shuffle the non-HADQAET texts
    random.shuffle(other_texts_here)

    # cut the shuffled list into three pieces
    # Example with eval_count = 3 and test_count = 3:
    #   positions 0, 1, 2   -> eval
    #   positions 3, 4, 5   -> test
    #   positions 6 onwards -> train
    eval_end = eval_count # eval stops here
    test_end = eval_count + test_count # test stops here
    eval_texts = other_texts_here[0:eval_end]
    test_texts = other_texts_here[eval_end:test_end]
    leftover_texts = other_texts_here[test_end:] # everything after test, will go into train

    # record the split for every text
    for text in eval_texts:
        split_of_text[text] = "eval"

    for text in test_texts:
        split_of_text[text] = "test"

    for text in leftover_texts:
        split_of_text[text] = "train"

    for text in hadqaet_texts_here:
        split_of_text[text] = "train"# HADQAET texts always go to train

    # Print a summary line for this stratum
    print(f"{stratum_name}: Total ({total_count}), Eval ({len(eval_texts)}), Test ({len(test_texts)}), Train ({len(leftover_texts) + len(hadqaet_texts_here)})")

angry + no slang: Total (9211), Eval (921), Test (921), Train (7369)
angry + slang: Total (346), Eval (35), Test (35), Train (276)
fear + no slang: Total (1391), Eval (139), Test (139), Train (1113)
fear + slang: Total (55), Eval (6), Test (6), Train (43)
happy + no slang: Total (6559), Eval (656), Test (656), Train (5247)
happy + slang: Total (183), Eval (18), Test (18), Train (147)
neutral + no slang: Total (7085), Eval (708), Test (708), Train (5669)
neutral + slang: Total (44), Eval (4), Test (4), Train (36)
sad + no slang: Total (6052), Eval (605), Test (605), Train (4842)
sad + slang: Total (141), Eval (14), Test (14), Train (113)
surprise + no slang: Total (2482), Eval (248), Test (248), Train (1986)
surprise + slang: Total (56), Eval (6), Test (6), Train (44)


In [ ]:
# Copy each text's split back onto every row with that text
new_split = []
for text in usual["text"]:
    new_split.append(split_of_text[text.strip()])
usual["split"] = new_split

## 6. Check the split

In [ ]:
# Check size of each split

n_train = (usual["split"] == "train").sum()
n_eval = (usual["split"] == "eval").sum()
n_test = (usual["split"] == "test").sum()

print(f"Train: {n_train} rows, {round(n_train / len(usual) * 100, 1)}%")
print(f"Eval: {n_eval} rows, {round(n_eval / len(usual) * 100, 1)}%")
print(f"Test: {n_test} rows, {round(n_test / len(usual) * 100, 1)}%")

Train: 27864 rows, 80.1%
Eval: 3457 rows, 9.9%
Test: 3445 rows, 9.9%


In [ ]:
# Check label mix per split (each column should look about the same)

labels = ["angry", "happy", "sad", "fear", "surprise", "neutral"]

print("label      train %   eval %   test %")
for label in labels:
    line = label
    for split in ["train", "eval", "test"]:
        rows_in_split = usual[usual["split"] == split]
        percent = (rows_in_split["label"] == label).sum() / len(rows_in_split) * 100
        line = f"{line}     {round(percent, 1)}"
    print(line) #sorry for ugly print

label      train %   eval %   test %
angry     30.1     29.7     29.5
happy     19.5     19.5     19.6
sad     17.9     18.0     18.0
fear     4.4     4.4     4.4
surprise     7.5     7.7     7.6
neutral     20.5     20.7     20.9


In [ ]:
# Check slang share per split

for split in ["train", "eval", "test"]:
    rows_in_split = usual[usual["split"] == split]
    n_slang = rows_in_split["has_slang_simple"].sum()
    percent = n_slang / len(rows_in_split) * 100
    print(f"{split}: slang posts: {n_slang} ({round(percent, 2)}%)")

train: slang posts: 702 (2.52%)
eval: slang posts: 94 (2.72%)
test: slang posts: 89 (2.58%)


In [ ]:
# Check the two most important rules for data leakage

# no HADQAET post in eval or test
hadqaet_rows = usual[usual["in_hadqaet"] == True]
hadqaet_outside_train = (hadqaet_rows["split"] != "train").sum()
print("HADQAET rows outside train (must be 0):", hadqaet_outside_train)

# no text appears in more than one split
splits_per_text = {}
for index, row in usual.iterrows():
    text = row["text"].strip()
    if text not in splits_per_text:
        splits_per_text[text] = set()
    splits_per_text[text].add(row["split"])

texts_in_several_splits = 0
for text in splits_per_text:
    if len(splits_per_text[text]) > 1:
        texts_in_several_splits = texts_in_several_splits + 1
print("Texts found in more than one split (must be 0):", texts_in_several_splits)


HADQAET rows outside train (must be 0): 0
Texts found in more than one split (must be 0): 0


## 7. Save

In [ ]:
columns_to_save = ["post_id", "split", "text", "label", "in_hadqaet", "has_slang_simple", "orig_split", "orig_id"]
output_path = os.path.join(processed_folder, "usual_splits.csv")
usual[columns_to_save].to_csv(output_path, index=False, encoding="utf-8-sig")

print("Saved", output_path)
usual[columns_to_save].head()

Saved /Users/cierrajagan/github/W266-slang-emotion-mt/data/processed/usual_splits.csv


,post_id,split,text,label,in_hadqaet,has_slang_simple,orig_split,orig_id
0,usual_train_00001,train,气死姐姐了，快二是阵亡了吗，尼玛，一个半小时过去了也没上车,angry,False,False,train,1
1,usual_train_00002,train,妞妞啊，今天又承办了一个发文登记文号是126~嘻~么么哒~晚安哟,happy,False,True,train,2
2,usual_train_00003,train,这里还值得注意另一个事实，就是张鞠存原有一个东溪草堂为其读书处。,neutral,False,False,train,3
3,usual_train_00004,train,这在前华约国家(尤其是东德)使用R-73的首次联合演习期间，被一些北约组织的飞行员所证实。,neutral,False,False,train,4
4,usual_train_00005,train,TinyThief上wii了？！,surprise,False,False,train,5
